In [1]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

In [2]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.optimizers import SGD
import random

In [3]:
words=[]
labels = []
documents = []
ignore_words = ['?', '!']


In [6]:
data_file = open('/content/intents.json').read()
intents = json.loads(data_file)

In [7]:
# FIX 1: Ensure all NLTK data is downloaded properly
import nltk
import ssl

# Handle SSL certificate issues (common in some environments)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [11]:

# Your code (now fixed)
import json

# Load your intents JSON
with open('/content/intents.json', 'r') as f:
    intents = json.load(f)

words = []
documents = []

for intent in intents['intents']:
    for pattern in intent['patterns']:
        # Tokenize each word
        w = nltk.word_tokenize(pattern)
        words.extend(w)

        # Add documents in the corpus
        documents.append((w, intent['tag']))

print(f"✓ Loaded {len(documents)} patterns")
print(f"✓ Found {len(set(words))} unique words")


✓ Loaded 113 patterns
✓ Found 145 unique words


In [12]:
import nltk
nltk.download('wordnet')
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
# sort labels
labels = sorted(list(set(labels)))
# documents = combination between patterns and intents
print (len(documents), "documents")
# labels = intents
print (len(labels), "labels", labels)
# words = all words, vocabulary
print (len(words), "unique lemmatized words", words)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


113 documents
31 labels ['Identity', 'activity', 'age', 'appreciate', 'contact', 'covid19', 'cricket', 'datetime', 'exclaim', 'goodbye', 'google', 'greeting', 'greetreply', 'haha', 'inspire', 'insult', 'jokes', 'karan', 'news', 'nicetty', 'no', 'noanswer', 'options', 'programmer', 'riddle', 'song', 'suggest', 'thanks', 'timer', 'weather', 'whatsup']
130 unique lemmatized words ["'m", "'s", ',', '10', '19', 'a', 'age', 'am', 'anyone', 'are', 'ask', 'awesome', 'bad', 'bbye', 'be', 'best', 'bye', 'can', 'contact', 'could', 'covid', 'creator', 'cricket', 'current', 'date', 'day', 'designed', 'developer', 'do', 'doing', 'dumb', 'fine', 'for', 'funny', 'get', 'good', 'goodbye', 'google', 'great', 'haha', 'he', 'hello', 'help', 'helpful', 'helping', 'hey', 'hi', 'hola', 'hot', 'how', 'i', 'idiot', 'india', 'inspiration', 'inspires', 'internet', 'is', 'it', 'joke', 'karan', 'know', 'later', 'latest', 'laugh', 'lmao', 'lol', 'lost', 'made', 'make', 'malik', 'match', 'me', 'motivates', 'namaste'

In [13]:
pickle.dump(words,open('/content/sample_data/words.pkl','wb'))
pickle.dump(labels,open('/content/sample_data/labels.pkl','wb'))

In [14]:
labels = [intent['tag'] for intent in intents['intents']]
print(labels)  # Check if 'google' is here

['google', 'greeting', 'goodbye', 'thanks', 'noanswer', 'options', 'jokes', 'Identity', 'datetime', 'whatsup', 'haha', 'programmer', 'insult', 'activity', 'exclaim', 'weather', 'karan', 'contact', 'appreciate', 'nicetty', 'no', 'news', 'inspire', 'cricket', 'song', 'greetreply', 'timer', 'covid19', 'suggest', 'riddle', 'age']


In [15]:
# Check what's actually in documents vs labels
for doc in documents:
    tag = doc[1]
    if tag not in labels:
        print(f"Found in documents but NOT in labels: '{tag}'")
        print(f"  Raw representation: {repr(tag)}")  # Shows hidden chars
        print(f"  Length: {len(tag)}")

In [18]:
# create our training data
training = []
# create an empty array for our output
output_empty = [0] * len(labels)
# training set, bag of words for each sentence
for doc in documents:
    # initialize our bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[labels.index(doc[1])] = 1  # ← This line was missing


    training.append([bag, output_row])
# shuffle our features and turn into np.array
random.shuffle(training)

train_x = [item[0] for item in training]
train_y = [item[1] for item in training]

train_x = np.array(train_x)
train_y = np.array(train_y)


In [19]:
# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [45]:
from tensorflow.keras.optimizers import SGD

sgd = SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=True
)

model.compile(
    loss='categorical_crossentropy',
    optimizer=sgd,
    metrics=['accuracy']
)



In [46]:



model.save('chatbot_model.h5')

print("model created")

model created


In [47]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import pickle
import numpy as np

In [48]:

import json
import random

In [49]:
def clean_up_sentence(sentence):
    # tokenize the pattern - split words into array
    sentence_words = nltk.word_tokenize(sentence)
    # stem each word - create short form for word
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

In [50]:
def bow(sentence, words, show_details=True):
    # tokenize the pattern
    sentence_words = clean_up_sentence(sentence)
    # bag of words - matrix of N words, vocabulary matrix
    bag = [0]*len(words)
    for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                # assign 1 if current word is in the vocabulary position
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)
    return(np.array(bag))


In [51]:
def predict_class(sentence, model):
    p = bow(sentence, words, show_details=False)

    res = model.predict(np.array([p]), verbose=0)[0]

    print("Raw prediction:", res)
    print("Max probability:", max(res))

    ERROR_THRESHOLD = 0.25

    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]

    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({
            "intent": labels[r[0]],
            "probability": str(r[1])
        })

    return return_list

In [52]:
def getResponse(ints, intents_json):
    if not ints:
        return "Sorry, I don't understand."

    tag = ints[0]['intent']

    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

    return "Sorry, I don't understand."

In [53]:
def chatbot_response(msg):
    ints = predict_class(msg, model)
    res = getResponse(ints, intents)
    return res

In [54]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        16,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 31)             │         2,015 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,039 (105.62 KB)

 Trainable params: 27,039 (105.62 KB)

 Non-trainable params: 0 (0.00 B)

In [57]:
print(model.metrics_names)
loss, acc = model.evaluate(train_x, train_y, verbose=0)
print("Training accuracy:", acc)
print("Training loss:", loss)

['loss', 'compile_metrics']
Training accuracy: 0.05309734493494034
Training loss: 3.430802583694458


In [58]:
hist = model.fit(
    train_x,
    train_y,
    epochs=200,
    batch_size=5,
    verbose=1
)

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0265 - loss: 3.4407
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0619 - loss: 3.3760
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0708 - loss: 3.3446     
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1150 - loss: 3.2497
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1327 - loss: 3.2114 
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1858 - loss: 3.1025     
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2301 - loss: 3.0046     
Epoch 8/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2566 - loss: 2.9262 
Epoch 9/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2035 - loss: 2.8839 
Epoch 10/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2655 - loss: 2.7252 
Epoch 11/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2832 - loss: 2.6046 
Epoch 12/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [59]:
loss, acc = model.evaluate(train_x, train_y, verbose=0)
print("Accuracy:", acc)

Accuracy: 0.991150438785553


In [61]:
kill_siri = False

while kill_siri != True:
    msg = input('You: ')

    if msg != '':

        res = chatbot_response(msg)
        print("Siri: " + res + '\n')

    if msg.lower() == 'bye':
        kill_siri = True

You: hi
Raw prediction: [1.7284756e-04 9.9494189e-01 1.6085219e-03 8.6191330e-06 1.7246587e-06
 2.1390467e-06 9.8958717e-06 2.8586835e-06 6.0863145e-05 9.1270613e-04
 1.0681824e-06 1.1322686e-06 7.3278985e-05 2.0381955e-05 1.2599766e-04
 6.2367588e-05 3.4583820e-05 2.0930887e-05 1.1915778e-04 2.9580040e-05
 4.0971505e-04 3.7776028e-06 7.4227455e-07 2.6544954e-05 1.4460683e-06
 1.4537080e-04 6.1202627e-06 9.8815467e-04 1.1891756e-04 9.3997363e-07
 8.7817702e-05]
Max probability: 0.9949419
Siri: Hello

You: how are you
Raw prediction: [3.0044108e-04 6.1623383e-01 7.6604178e-05 8.1564009e-05 1.0178806e-05
 2.9954061e-04 3.1194574e-04 1.3875208e-04 8.0067854e-05 3.7291822e-01
 1.3364688e-05 6.9558453e-05 6.8525912e-04 8.7034889e-04 8.3832725e-05
 3.4868997e-04 1.0165167e-05 6.3862117e-06 1.3816099e-03 2.6838522e-04
 3.2668337e-05 7.0281065e-05 4.1780031e-06 3.9605406e-05 3.2054475e-05
 4.7326527e-04 9.1315131e-05 4.8215495e-04 9.9807407e-04 6.9140106e-06
 3.5807863e-03]
Max probability: 0.

KeyboardInterrupt: Interrupted by user